# Tarea 2 · Parte 2 — De los modelos base a la extracción estructurada

Alexander Noblejas (`@AlexNoblejas`) · Python Intermedio, Diplomatura QLAB

Línea base, extracción de campos con un LLM, auditoría humana y comparación contra un segundo
modelo por modalidad, todo sobre las mismas diez muestras de cada tipo.

In [ ]:
import json, os, re, time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

import corpus_audio, voz

AQUI = Path.cwd()
DATOS = AQUI / "datos"

documentos = pd.read_csv(DATOS / "documentos.csv").fillna({"texto_ocr": "", "texto_referencia": ""})
transcripciones = pd.read_csv(DATOS / "transcripciones.csv").fillna({"texto_asr": "", "texto_referencia": ""})
print(f"documentos.csv      {documentos.shape}")
print(f"transcripciones.csv {transcripciones.shape}")
print(f"referencias manuales: {(documentos.texto_referencia.str.strip() != '').sum()} documentos, "
      f"{(transcripciones.texto_referencia.str.strip() != '').sum()} clips")

## 1. Muestra de evaluación

Diez documentos y diez clips **del corpus de la Parte 1**, sin recolectar nada nuevo. La
transcripción de referencia la hizo el estudiante a mano, escuchando el audio y mirando la
imagen, **sin ver la salida del OCR ni del ASR**: si la referencia se contagia de lo que
escribió el modelo, el CER y el WER quedan artificialmente bajos.

Dos decisiones metodológicas que conviene dejar por escrito:

- **La muestra se fijó antes de correr cualquier modelo.** Los ids se eligieron el mismo día
  en que se armó el corpus, antes de que existiera una sola métrica, así que no hubo manera de
  seleccionar casos favorables ni de ajustar parámetros mirando resultados.
- **Cubre los tres tipos de documento y cinco condiciones de grabación**: entrevista de calle,
  testimonio leído en interior, conferencia de prensa, testimonio largo de sierra y marcha con
  multitud y música. Se privilegiaron los clips cortos de cada condición para que la
  transcripción manual fuera viable sin sacrificar variedad.

In [ ]:
MUESTRA_DOC = documentos[documentos.texto_referencia.str.strip() != ""].id.tolist()
MUESTRA_AUDIO = transcripciones[transcripciones.texto_referencia.str.strip() != ""].id.tolist()

print(f"{len(MUESTRA_DOC)} documentos: {MUESTRA_DOC}")
print(f"{len(MUESTRA_AUDIO)} clips: {MUESTRA_AUDIO}")

tipos = documentos.set_index("id").loc[MUESTRA_DOC, "tipo"].value_counts()
grupos = transcripciones.set_index("id").loc[MUESTRA_AUDIO, "grupo"].value_counts()
print(f"\ntipos de documento en la muestra: {dict(tipos)}")
print(f"condiciones de grabación en la muestra: {dict(grupos)}")

assert len(MUESTRA_DOC) == 10, f"la muestra de documentos tiene {len(MUESTRA_DOC)}, debe tener 10"
assert len(MUESTRA_AUDIO) == 10, f"la muestra de audio tiene {len(MUESTRA_AUDIO)}, debe tener 10"
assert {"impreso", "foto", "manuscrito"} <= set(tipos.index), "la muestra no cubre los tres tipos"
print("\nMuestra conforme al enunciado.")

## 2. Línea base, sin ajuste

La línea base que fija el enunciado es **PaddleOCR sin personalización adicional** y
**faster-whisper `small` en CPU**, y es exactamente la configuración con la que corrió la
Parte 1: los textos que están en los CSV ya son la salida de esa configuración, así que las
métricas se calculan sobre ellos sin volver a inferir.

Referencia e hipótesis se normalizan con el mismo criterio (`voz.normalizar`: minúsculas, sin
puntuación, espacios colapsados, tildes conservadas) y en el WER se eliminan las marcas
`[HABLANTE_n]`, que no son palabras habladas.

In [ ]:
def sin_marcas(texto: str) -> str:
    """Quita las marcas de hablante: no son palabras dichas y no deben contar en el WER."""
    return re.sub(r"\[HABLANTE_\d+\]|\[H\d+\]|^H\d+:", " ", str(texto), flags=re.MULTILINE)


def metrica_por_fila(df, col_hip, col_ref, funcion, limpiar=None):
    """Métrica individual por fila, para poder mirar los casos y no solo el agregado."""
    salida = []
    for r in df.itertuples():
        ref, hip = getattr(r, col_ref), getattr(r, col_hip)
        if limpiar:
            hip, ref = limpiar(hip), limpiar(ref)
        salida.append(round(funcion([ref], [hip]), 4) if str(ref).strip() else np.nan)
    return salida


doc_eval = documentos[documentos.id.isin(MUESTRA_DOC)].copy()
aud_eval = transcripciones[transcripciones.id.isin(MUESTRA_AUDIO)].copy()

doc_eval["cer_paddleocr"] = metrica_por_fila(doc_eval, "texto_ocr", "texto_referencia", voz.cer)
aud_eval["wer_whisper_small"] = metrica_por_fila(aud_eval, "texto_asr", "texto_referencia",
                                                 voz.wer, limpiar=sin_marcas)

CER_BASE = voz.cer(doc_eval.texto_referencia.tolist(), doc_eval.texto_ocr.tolist())
WER_BASE = voz.wer([sin_marcas(t) for t in aud_eval.texto_referencia],
                   [sin_marcas(t) for t in aud_eval.texto_asr])

print(f"LÍNEA BASE")
print(f"  CER PaddleOCR (10 documentos): {CER_BASE:.4f}")
print(f"  WER faster-whisper small (10 clips): {WER_BASE:.4f}")
print()
print(doc_eval[["id", "tipo", "cer_paddleocr"]].to_string(index=False))
print()
print(aud_eval[["id", "grupo", "duracion_seg", "wer_whisper_small"]].to_string(index=False))

In [ ]:
# El agregado esconde lo interesante: el error por tipo de documento y por condición de audio
fig = px.bar(doc_eval.sort_values("cer_paddleocr"), x="id", y="cer_paddleocr", color="tipo",
             title=f"El CER se concentra en un tipo de documento: la media de {CER_BASE:.2f} "
                   f"no describe a ninguno de los tres",
             labels={"id": "Documento", "cer_paddleocr": "CER", "tipo": "Tipo"})
fig.update_layout(title_font_size=13)
fig.show()
print(doc_eval.groupby("tipo").cer_paddleocr.agg(["count", "mean", "min", "max"]).round(3))
print()
print(aud_eval.groupby("grupo").wer_whisper_small.agg(["count", "mean", "min", "max"]).round(3))

## 3. Extracción estructurada con un LLM

Se extraen campos JSON de cada texto del corpus con `llama-3.3-70b-versatile` vía Groq. El
enunciado permite API explícitamente ("Ollama, Qwen local, Gemini API o similar").

La clave se lee de la variable de entorno `GROQ_API_KEY` y **no se guarda en el repositorio**,
que es público y donde el propio enunciado prohíbe subir credenciales.

Campos obligatorios: `modalidad`, `tema`, `resumen`. En documentos se piden `emisor`, `fecha` y
`monto` **solo si aparecen en el texto**; en audios, `citas_clave` y `numero_hablantes` cuando
se puedan sustentar. Los ausentes van como `null` y el prompt prohíbe inventar campos.

In [ ]:
from groq import Groq

MODELO_LLM = "llama-3.3-70b-versatile"
cliente = Groq()          # toma la clave de GROQ_API_KEY; nunca se escribe en disco

INSTRUCCIONES = """Eres un asistente que extrae campos estructurados de textos peruanos.
Devuelve SOLO un objeto JSON. Reglas estrictas:
- Campos siempre presentes: "modalidad" ("documento" o "audio"), "tema" (3 a 8 palabras),
  "resumen" (una o dos oraciones, en español).
- Si la modalidad es documento, incluye "emisor", "fecha" y "monto" SOLO si aparecen
  literalmente en el texto; si no aparecen, usa null.
- Si la modalidad es audio, incluye "citas_clave" (lista de 1 a 3 frases textuales breves) y
  "numero_hablantes" (entero) SOLO si se pueden sustentar con el texto; si no, null.
- No inventes datos y no agregues campos que no estén en esta lista.
- El texto viene de OCR o de transcripción automática y puede traer errores: no los corrijas
  inventando, trabaja con lo que dice."""


def extraer(texto: str, modalidad: str, intentos: int = 3) -> dict:
    mensaje = f"modalidad: {modalidad}\n\ntexto:\n{str(texto)[:6000]}"
    for i in range(intentos):
        try:
            r = cliente.chat.completions.create(
                model=MODELO_LLM, temperature=0.0,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": INSTRUCCIONES},
                          {"role": "user", "content": mensaje}])
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            if i == intentos - 1:
                return {"error": f"{type(e).__name__}: {e}"}
            time.sleep(2 * (i + 1))


tareas = ([(r.id, "documento", r.texto_ocr) for r in documentos.itertuples() if str(r.texto_ocr).strip()] +
          [(r.id, "audio", r.texto_asr) for r in transcripciones.itertuples() if str(r.texto_asr).strip()])
print(f"{len(tareas)} textos a procesar con {MODELO_LLM}")

filas_llm, t0 = [], time.perf_counter()
for i, (cid, modalidad, texto) in enumerate(tareas, 1):
    campos = extraer(texto, modalidad)
    filas_llm.append({"id": cid, "modalidad": modalidad,
                      "campos_json": json.dumps(campos, ensure_ascii=False),
                      "modelo_llm": f"{MODELO_LLM} (Groq)", "fecha": date.today().isoformat()})
    if i % 10 == 0 or i == len(tareas):
        print(f"  {i}/{len(tareas)} · {time.perf_counter() - t0:.0f} s", flush=True)

extraccion = pd.DataFrame(filas_llm)
extraccion.to_csv(DATOS / "extraccion_llm.csv", index=False)
fallos = extraccion.campos_json.str.contains('"error"').sum()
print(f"\nextraccion_llm.csv {extraccion.shape} · {fallos} fallos")
print(json.dumps(json.loads(extraccion.campos_json.iloc[0]), ensure_ascii=False, indent=2)[:600])

## 4. Auditoría humana de 20 extracciones

Veinte extracciones al azar —diez documentos y diez audios— revisadas a mano. El criterio de
corrección se fija antes de mirar los casos:

Una extracción cuenta como **correcta** solo si se cumplen las cuatro condiciones:

1. `tema` describe efectivamente el contenido del texto.
2. `resumen` no afirma nada que el texto no diga.
3. Ningún campo opcional está relleno con información que no aparece en el texto (ese es el
   error grave: un `monto` o un `emisor` inventados).
4. Los campos que sí aparecen en el texto no están puestos como `null` sin razón.

La primera ejecución de esta celda escribe `auditoria_humana.xlsx` con los veinte casos y se
detiene. Se llena a mano y se vuelve a ejecutar el notebook.

In [ ]:
AUDITORIA = AQUI / "auditoria_humana.xlsx"
N_AUDITORIA = 20

sorteo = pd.concat([
    extraccion[extraccion.modalidad == "documento"].sample(10, random_state=20261025),
    extraccion[extraccion.modalidad == "audio"].sample(10, random_state=20261025),
])

if not AUDITORIA.exists():
    textos = {**{r.id: r.texto_ocr for r in documentos.itertuples()},
              **{r.id: r.texto_asr for r in transcripciones.itertuples()}}
    revision = pd.DataFrame({
        "id": sorteo.id,
        "modalidad": sorteo.modalidad,
        "campos_extraidos": [json.dumps(json.loads(c), ensure_ascii=False, indent=1)
                             for c in sorteo.campos_json],
        "texto_fuente": [str(textos.get(i, ""))[:1200] for i in sorteo.id],
        "correcto_si_no": "",
        "error_observado": "",
    })
    with pd.ExcelWriter(AUDITORIA) as w:
        revision.to_excel(w, index=False, sheet_name="auditoria")
    print(f"Escrito {AUDITORIA.name} con {len(revision)} casos.")
    print("Llena las columnas 'correcto_si_no' (si/no) y 'error_observado', guarda, y vuelve a ejecutar.")
else:
    print(f"{AUDITORIA.name} ya existe: se usa lo que tiene.")

In [ ]:
aud = pd.read_excel(AUDITORIA)
aud["correcto_si_no"] = aud.correcto_si_no.astype(str).str.strip().str.lower()
juzgados = aud[aud.correcto_si_no.isin(["si", "sí", "no"])]

if len(juzgados) < N_AUDITORIA:
    ACUERDO = np.nan
    print(f"AUDITORÍA PENDIENTE: {len(juzgados)} de {N_AUDITORIA} casos juzgados.")
else:
    correctas = juzgados.correcto_si_no.isin(["si", "sí"]).sum()
    ACUERDO = correctas / N_AUDITORIA
    print(f"ACUERDO LLM–HUMANO: {correctas}/{N_AUDITORIA} = {ACUERDO:.0%}")
    print(f"\n  por modalidad:")
    print(juzgados.assign(ok=juzgados.correcto_si_no.isin(["si", "sí"]))
          .groupby("modalidad").ok.agg(["sum", "count"]).to_string())
    errores = juzgados[~juzgados.correcto_si_no.isin(["si", "sí"])]
    print(f"\n  ejemplos de error ({len(errores)}):")
    for r in errores.head(4).itertuples():
        print(f"    {r.id} ({r.modalidad}): {str(r.error_observado)[:160]}")

## 5. Comparación de modelos, sobre las mismas muestras

### Documentos: PaddleOCR frente a un segundo lector

El segundo OCR es **EasyOCR** (detector CRAFT + reconocedor CRNN), que es una arquitectura
distinta a la de PaddleOCR y se instala con pip, sin instalador de sistema. Si Tesseract está
disponible en el PATH se usa ese, que es el ejemplo que da el enunciado.

Se mide CER **sobre los mismos diez documentos** y el tiempo por página.

In [ ]:
import shutil

SEGUNDO_OCR, leer_segundo = None, None

if shutil.which("tesseract"):
    import pytesseract
    from PIL import Image
    SEGUNDO_OCR = f"Tesseract {pytesseract.get_tesseract_version()} (lang=spa)"
    def leer_segundo(ruta):
        return pytesseract.image_to_string(Image.open(ruta), lang="spa")
else:
    try:
        import easyocr
        lector = easyocr.Reader(["es"], gpu=False)
        SEGUNDO_OCR = "EasyOCR 1.7 (CRAFT + CRNN, lang=es, CPU)"
        def leer_segundo(ruta):
            return "\n".join(lector.readtext(str(ruta), detail=0, paragraph=True))
    except Exception as e:
        print(f"No hay segundo OCR disponible: {type(e).__name__}: {e}")

if leer_segundo:
    print(f"Segundo OCR: {SEGUNDO_OCR}\n")
    textos2, tiempos2 = [], []
    for r in doc_eval.itertuples():
        t0 = time.perf_counter()
        try:
            texto = leer_segundo(AQUI / r.archivo)
        except Exception as e:
            texto = ""
            print(f"  {r.id}: falló ({type(e).__name__})")
        tiempos2.append(round(time.perf_counter() - t0, 2))
        textos2.append(texto)
        print(f"  {r.id:12s} {len(texto):5d} caracteres  {tiempos2[-1]:6.1f} s", flush=True)

    doc_eval["texto_ocr2"] = textos2
    doc_eval["segundos_ocr2"] = tiempos2
    doc_eval["cer_ocr2"] = metrica_por_fila(doc_eval, "texto_ocr2", "texto_referencia", voz.cer)
    CER_OCR2 = voz.cer(doc_eval.texto_referencia.tolist(), doc_eval.texto_ocr2.tolist())
    print(f"\n  CER {SEGUNDO_OCR}: {CER_OCR2:.4f}   vs   PaddleOCR: {CER_BASE:.4f}")
    print()
    print(doc_eval[["id", "tipo", "cer_paddleocr", "cer_ocr2", "segundos_ocr", "segundos_ocr2"]]
          .to_string(index=False))
else:
    CER_OCR2 = np.nan

### Audio: faster-whisper frente a otro ASR

El enunciado pide una familia distinta, no otro tamaño de Whisper: se usa **Qwen3-ASR-0.6B**,
que `voz.py` ya tiene en su catálogo y que se descarga de Hugging Face sin acceso restringido.
Se mide WER y latencia **sobre los mismos diez clips**, y el factor de tiempo real como tiempo
de proceso dividido por duración del audio.

In [ ]:
SEGUNDO_ASR, transcriptor2 = None, None
try:
    t0 = time.perf_counter()
    transcriptor2 = voz.Transcriptor("qwen3-asr-0.6b", device="cpu")
    SEGUNDO_ASR = f"{transcriptor2.info['nombre']} (CPU)"
    print(f"{SEGUNDO_ASR} · cargado en {time.perf_counter() - t0:.1f} s")
except Exception as e:
    print(f"Qwen3-ASR no disponible: {type(e).__name__}: {e}")

if transcriptor2:
    textos2, tiempos2 = [], []
    for r in aud_eval.itertuples():
        audio = voz.cargar_audio(AQUI / r.archivo_audio)
        t0 = time.perf_counter()
        try:
            segs = transcriptor2.segmentos(audio, idioma="es")
            texto = voz.a_texto(segs)
        except Exception as e:
            texto = ""
            print(f"  {r.id}: falló ({type(e).__name__}: {e})")
        tiempos2.append(round(time.perf_counter() - t0, 2))
        textos2.append(texto)
        print(f"  {r.id:12s} {r.duracion_seg:6.1f} s audio  {tiempos2[-1]:7.1f} s proceso  "
              f"x{tiempos2[-1] / r.duracion_seg:.2f}", flush=True)

    aud_eval["texto_asr2"] = textos2
    aud_eval["segundos_asr2"] = tiempos2
    aud_eval["factor_asr2"] = (aud_eval.segundos_asr2 / aud_eval.duracion_seg).round(3)
    aud_eval["wer_asr2"] = metrica_por_fila(aud_eval, "texto_asr2", "texto_referencia",
                                            voz.wer, limpiar=sin_marcas)
    WER_ASR2 = voz.wer([sin_marcas(t) for t in aud_eval.texto_referencia],
                       [sin_marcas(t) for t in aud_eval.texto_asr2])
    print(f"\n  WER {SEGUNDO_ASR}: {WER_ASR2:.4f}   vs   faster-whisper small: {WER_BASE:.4f}")
    print()
    print(aud_eval[["id", "grupo", "duracion_seg", "wer_whisper_small", "wer_asr2",
                    "factor_tiempo_real", "factor_asr2"]].to_string(index=False))
else:
    WER_ASR2 = np.nan

## 6. Casos difíciles

Los manuscritos en documentos, y la marcha con multitud y música en audio. Son los dos sitios
donde se ve si el pipeline aguanta algo distinto de una página limpia y una voz sola.

In [ ]:
print("=== DOCUMENTOS: CER por tipo ===")
tabla_doc = doc_eval.groupby("tipo").agg(
    n=("id", "size"), cer_paddle=("cer_paddleocr", "mean"),
    **({"cer_ocr2": ("cer_ocr2", "mean")} if "cer_ocr2" in doc_eval else {})).round(3)
print(tabla_doc.to_string())

print("\n=== AUDIO: WER por condición de grabación ===")
tabla_aud = aud_eval.groupby("grupo").agg(
    n=("id", "size"), segundos=("duracion_seg", "sum"), wer_small=("wer_whisper_small", "mean"),
    **({"wer_asr2": ("wer_asr2", "mean")} if "wer_asr2" in aud_eval else {})).round(3)
print(tabla_aud.to_string())

peor = aud_eval.nlargest(1, "wer_whisper_small").iloc[0]
print(f"\nPeor clip: {peor.id} ({peor.grupo}), WER {peor.wer_whisper_small:.2f}")
print(f"  referencia: {peor.texto_referencia[:220]}")
print(f"  hipótesis : {peor.texto_asr[:220]}")

## 7. Comparativa y decisión

Tabla de los cuatro modelos sobre las mismas muestras, y la elección para un entorno real
considerando precisión, costo computacional y latencia.

In [ ]:
comparativa = pd.DataFrame([
    {"modalidad": "documentos", "modelo": "PaddleOCR 3.7 (base, transformers, CPU)",
     "metrica": "CER", "valor": round(CER_BASE, 4),
     "seg_por_unidad": round(doc_eval.segundos_ocr.mean(), 1)},
    {"modalidad": "documentos", "modelo": SEGUNDO_OCR or "segundo OCR no disponible",
     "metrica": "CER", "valor": round(CER_OCR2, 4) if CER_OCR2 == CER_OCR2 else np.nan,
     "seg_por_unidad": round(doc_eval.segundos_ocr2.mean(), 1) if "segundos_ocr2" in doc_eval else np.nan},
    {"modalidad": "audio", "modelo": "faster-whisper small (base, CPU int8)",
     "metrica": "WER", "valor": round(WER_BASE, 4),
     "seg_por_unidad": round(aud_eval.factor_tiempo_real.mean(), 2)},
    {"modalidad": "audio", "modelo": SEGUNDO_ASR or "segundo ASR no disponible",
     "metrica": "WER", "valor": round(WER_ASR2, 4) if WER_ASR2 == WER_ASR2 else np.nan,
     "seg_por_unidad": round(aud_eval.factor_asr2.mean(), 2) if "factor_asr2" in aud_eval else np.nan},
])
comparativa.to_csv(DATOS / "comparativa_modelos.csv", index=False)
print(comparativa.to_string(index=False))
print(f"\nAcuerdo LLM–humano: {ACUERDO:.0%}" if ACUERDO == ACUERDO else "\nAuditoría pendiente")

RESULTADOS = {"CER": round(CER_BASE, 4), "WER": round(WER_BASE, 4),
              "CER_segundo_ocr": None if CER_OCR2 != CER_OCR2 else round(CER_OCR2, 4),
              "WER_segundo_asr": None if WER_ASR2 != WER_ASR2 else round(WER_ASR2, 4),
              "acuerdo_llm_humano": None if ACUERDO != ACUERDO else round(ACUERDO, 2),
              "n_documentos": int(len(documentos)), "n_clips": int(len(transcripciones)),
              "minutos_audio": round(transcripciones.duracion_seg.sum() / 60, 1),
              "modelo_ocr": "PaddleOCR 3.7", "modelo_asr": "faster-whisper small",
              "modelo_llm": MODELO_LLM, "segundo_ocr": SEGUNDO_OCR, "segundo_asr": SEGUNDO_ASR}
(DATOS / "resultados.json").write_text(json.dumps(RESULTADOS, ensure_ascii=False, indent=2),
                                       encoding="utf-8")
print(f"\nresultados.json escrito: lo lee el dashboard")
RESULTADOS

In [ ]:
# Guardar la muestra evaluada con todas sus métricas: la usa el explorador del dashboard
doc_eval.to_csv(DATOS / "evaluacion_documentos.csv", index=False)
aud_eval.to_csv(DATOS / "evaluacion_audio.csv", index=False)
print("evaluacion_documentos.csv", doc_eval.shape)
print("evaluacion_audio.csv     ", aud_eval.shape)

### Qué elegiría para un entorno real

*Se completa con los números de arriba una vez ejecutado el notebook.*

**Por tipo de documento y por tipo de audio:** ver las tablas de la sección 6. El patrón que
hay que explicar es dónde se concentra el error y por qué — tipografía y trazo en los
manuscritos, superposición de voces y música en la marcha.